# Práctica 3.1 — Entrenar y servir un bot sencillo (Uma Smash Derby)

CartPole-v1 es el sustituto didáctico del bot CPU de Uma Smash: equilibrar el poste se parece a mantener a la luchadora sobre el escenario.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno → **GPU T4**.

In [ ]:
!pip install -q gymnasium stable-baselines3
!nvidia-smi --query-gpu=name --format=csv

In [ ]:
import gymnasium as gym, time, numpy as np, torch
from stable_baselines3 import PPO

def entrenar(device):
    env = gym.make('CartPole-v1')
    modelo = PPO('MlpPolicy', env, verbose=0, device=device)
    inicio = time.time()
    modelo.learn(total_timesteps=100_000)
    t = time.time() - inicio
    print(f'Entrenamiento en {device}: {t:.1f} s')
    return modelo, env, t

modelo_cpu, env, t_cpu = entrenar('cpu')
modelo_cpu.save('bot_uma_smash')

In [ ]:
print('GPU disponible:', torch.cuda.is_available())
modelo_gpu, _, t_gpu = entrenar('cuda')

In [ ]:
# Latencia de inferencia en CPU
obs, _ = env.reset()
t = time.perf_counter()
for _ in range(10_000):
    accion, _ = modelo_cpu.predict(obs, deterministic=True)
lat_ms = (time.perf_counter() - t) / 10_000 * 1000
dps = 1000 / lat_ms
print(f'Latencia media: {lat_ms:.3f} ms -> {dps:,.0f} decisiones/s por nucleo')

In [ ]:
# Dimensionamiento: 2 000 bots a 10 Hz (fórmula de la guía) + 30 % de margen
import math
bots, hz = 2000, 10
decisiones = bots * hz
vcpu = decisiones / dps
vcpu_margen = math.ceil(vcpu * 1.3)
print(f'Decisiones/s necesarias: {decisiones:,}')
print(f'vCPU sin margen: {vcpu:.2f}  ->  con 30 % de margen: {vcpu_margen} vCPU')

import pandas as pd
tabla = pd.DataFrame({
  'Métrica': ['Entrenamiento CPU (s)', 'Entrenamiento GPU T4 (s)', 'Latencia inferencia CPU (ms)',
              'Decisiones/s por núcleo', 'vCPU para 2 000 bots a 10 Hz (+30 %)'],
  'Valor': [f'{t_cpu:.1f}', f'{t_gpu:.1f}', f'{lat_ms:.3f}', f'{dps:,.0f}', vcpu_margen]})
tabla

## Costo estimado por hora

Busca en la [calculadora de Google Cloud](https://cloud.google.com/products/calculator) una máquina **c3-standard** o **e2-standard** con las vCPU calculadas arriba (región us-central1) y anota el precio por hora.

## ¿Por qué ganó la CPU (o la GPU)?

Con una red tan pequeña como la de CartPole, copiar datos a la GPU cuesta más que el cálculo, así que normalmente la CPU es igual o más rápida. La GPU rinde con redes grandes, imágenes o muchos entornos en paralelo.